# 7.6節：誤差逆伝播法

バージョン1.10.10で動作確認しています．

事前にインストールが必要なパッケージ
+ Flux
+ Statistics
+ ProgressMeter
+ Plots

In [ ]:
using Flux

In [ ]:
# モデル定義（中間層なし），活性化関数はReLU
model = Chain(Dense(2 => 3, relu))

# 損失関数（モデルとデータを引数に取る）
loss(m, x, y) = sum((m(x) .- y).^2)

# Float32型でデータを生成（必須ではないが，Flux利用時は標準的）
x = randn(Float32, 2)
y_true = randn(Float32, 3)

# 勾配を計算
grads = gradient(m -> loss(m, x, y_true), model)

# パラメータごとに表示
for (p, g) in zip(Flux.trainable(model), grads)
    println("Param: ", p)
    println("Grad: ", g)
end

## XORの学習（上のプログラムとは独立に実行してください）

In [ ]:
using Flux, Statistics, ProgressMeter
device(x) = x  # CPU環境用に定義（GPU対応時はここを書き換える）

In [ ]:
# XOR のデータ生成
noisy = rand(Float32, 2, 1000) # 2×1000 Matrix{Float32}
truth = [xor(col[1] > 0.5, col[2] > 0.5) for col in eachcol(noisy)]  
    # 1000-element Vector{Bool}

In [ ]:
# モデル定義（CPU）
model = Chain(
    Dense(2 => 3, tanh),
    BatchNorm(3),
    Dense(3 => 2)
) |> device     

In [ ]:
# 初期出力を確認
out1 = model(noisy |> device)   # CPU の Matrix{Float32}
probs1 = softmax(out1)          # CPU のまま

In [ ]:
# モデルの学習のために，ワンホットエンコードされた正解ラベル（ある種の前処理）を64サンプルからなるバッチを利用
target = Flux.onehotbatch(truth, [true, false])  # 2×1000 OneHotMatrix
loader = Flux.DataLoader((noisy, target), batchsize=64, shuffle=true) # バッチの利用

In [ ]:
# 最適化手法の設定：Adamを学習率0.01で使用
opt_state = Flux.setup(Flux.Adam(0.01), model)

# 学習ループ
losses = []
@showprogress for epoch in 1:1_000
    for xy in loader
        x, y = xy |> device   # CPU-only なので x, y はそのまま

        loss_val, grads = Flux.withgradient(model) do m
            y_hat = m(x)
            Flux.logitcrossentropy(y_hat, y)
        end

        Flux.update!(opt_state, model, grads[1])
        push!(losses, loss_val)
    end
end

In [ ]:
# 学習後の評価
out2 = model(noisy |> device)
probs2 = softmax(out2)
mean((probs2[1, :] .> 0.5) .== truth)

In [ ]:
# 可視化
using Plots

p_true = scatter(noisy[1,:], noisy[2,:], zcolor=truth, title="True classification", legend=false)
p_raw =  scatter(noisy[1,:], noisy[2,:], zcolor=probs1[1,:], title="Untrained network", legend=false, clims=(0,1))
p_done = scatter(noisy[1,:], noisy[2,:], zcolor=probs2[1,:], title="Trained network", label="")

plot(p_true, p_raw, p_done, layout=(1,3), size=(1000,330))

In [ ]:
plot(losses; xaxis=(:log10, "iteration"),
    yaxis="loss", label="per batch")
n = length(loader)
plot!(n:n:length(losses), mean.(Iterators.partition(losses, n)),
    label="epoch mean", dpi=200)